# Stock Price Prediction using Deep Learning (LSTM)

This project uses an LSTM model to predict stock prices based on historical data.
It also helps in making Buy/Sell decisions based on predicted trends.

In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM

ModuleNotFoundError: No module named 'numpy'

In [9]:
df = pd.read_csv("/content/NFLX.csv")
df.head()

NameError: name 'pd' is not defined

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)

data = df[['Close']].values

scaler = MinMaxScaler(feature_range=(0,1))
scaled_data = scaler.fit_transform(data)

In [ ]:
df['MA50'] = df['Close'].rolling(window=50).mean()

In [ ]:
def create_dataset(data, time_step=60):
    X, y = [], []
    for i in range(len(data)-time_step-1):
        X.append(data[i:(i+time_step)])
        y.append(data[i+time_step][3])  # Close price is at index 3 in the multi-feature scaled_data
    return np.array(X), np.array(y)

X, y = create_dataset(scaled_data)

In [ ]:
delta = df['Close'].diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.rolling(window=14).mean()
avg_loss = loss.rolling(window=14).mean()

rs = avg_gain / avg_loss
df['RSI'] = 100 - (100 / (1 + rs))

In [ ]:
features = ['Open', 'High', 'Low', 'Close', 'Volume', 'MA50', 'RSI']
data = df[features].dropna()

scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(data)

In [ ]:
train_size = int(len(X) * 0.8)

X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

In [ ]:
model = Sequential()

model.add(LSTM(64, return_sequences=True, input_shape=(X.shape[1], X.shape[2])))
model.add(LSTM(64))
model.add(Dense(25))
model.add(Dense(1))

model.compile(optimizer='adam', loss='mean_squared_error')

In [ ]:
model.fit(X_train, y_train, epochs=20, batch_size=32, validation_data=(X_test, y_test))

In [ ]:
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

train_pred = scaler.inverse_transform(train_pred)
test_pred = scaler.inverse_transform(test_pred)

y_test_actual = scaler.inverse_transform(y_test.reshape(-1,1))

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test_actual, test_pred))
mae = mean_absolute_error(y_test_actual, test_pred)

print("RMSE:", rmse)
print("MAE:", mae)

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(y_test_actual, label='Actual')
plt.plot(test_pred, label='Predicted')
plt.title("Actual vs Predicted Stock Prices")
plt.legend()
plt.show()

In [ ]:
latest_actual = y_test_actual[-1][0]
latest_pred = test_pred[-1][0]

change = (latest_pred - latest_actual) / latest_actual

if change > 0.03:
    print("🔥 STRONG BUY")
elif change > 0.01:
    print("📈 BUY")
elif change < -0.03:
    print("🚨 STRONG SELL")
elif change < 0:
    print("📉 SELL")
else:
    print("⏸ HOLD")

In [ ]:
future_days = 7
future_input = scaled_data[-60:]

future_predictions = []

for _ in range(future_days):
    pred = model.predict(future_input.reshape(1,60,X.shape[2]))
    future_predictions.append(pred[0][0])

    new_row = np.append(future_input[-1][:-1], pred[0][0])
    future_input = np.vstack([future_input[1:], new_row])